In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.adapter.api.routers import student_assessment
from src.adapter.api.template.comment import AssessmentComment, Comment

In [2]:
from pathlib import Path

folder_path = Path("../raw_data/Ho_ba_hoc_sinh")

excel_files = [str(f) for f in folder_path.glob("*.xlsx")]

print(excel_files)

['..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_11_VOMINHKHANG_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_12_PHAMNHATDANGKHOA_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_13_TRANHOANGKY_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_14_NGUYENBINHPHUONGLAM_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_15_THAIPHUONGLINH_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_16_HONGUYENKHANHNGAN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_17_NGUYENNGOCKIMNGAN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_18_LUUGIANGHI_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_19_NGUYENBAONGOC_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_1_TRUONGPHUCAN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_20_NGUYENPHAMANNHIEN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_21_PHAMANNHIEN_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\HOC_BA_22_VUONGTAMNHU

In [3]:
import json
from pathlib import Path
from openpyxl import load_workbook


TARGET_SHEET = "NL, PC Mot 1 (2024-2025)"
INFO_SHEET = "Thông tin học sinh"


def norm(v):
    return "" if v is None else str(v).strip()


def find_exact_text(ws, target):
    target = target.strip().lower()
    for row in ws.iter_rows():
        for cell in row:
            if norm(cell.value).lower() == target:
                return cell.row, cell.column
    return None


def find_contains_text(ws, target):
    target = target.strip().lower()
    for row in ws.iter_rows():
        for cell in row:
            if target in norm(cell.value).lower():
                return cell.row, cell.column
    return None


def get_name(wb):
    if INFO_SHEET not in wb.sheetnames:
        return ""

    ws = wb[INFO_SHEET]

    # Tìm đúng nhãn "Họ và tên học sinh:"
    pos = find_contains_text(ws, "Họ và tên học sinh")
    if not pos:
         ""

    row, col = pos

    # Thử lấy các ô bên phải cùng hàng
    for c in range(1, min(col + 8, ws.max_column + 1)):
        value = norm(ws.cell(row, c).value)
        if value:
            return value.replace("Họ và tên học sinh: ", "").lower().title()

    # # Nếu ô bên phải không có, thử dòng dưới
    # for r in range(row + 1, min(row + 3, ws.max_row + 1)):
    #     for c in range(col, min(col + 8, ws.max_column + 1)):
    #         value = norm(ws.cell(r, c).value)
    #         if value:
    #             return value

    # return ""


def find_comment_column(ws, start_row):
    # Tìm cột "Nhận xét" gần nhất phía trên
    for r in range(start_row, 0, -1):
        for c in range(1, ws.max_column + 1):
            if norm(ws.cell(r, c).value).lower() == "nhận xét":
                return c
    return None


def get_comment_by_label(ws, label):
    pos = find_exact_text(ws, label)
    if not pos:
        return ""

    row, _ = pos
    comment_col = find_comment_column(ws, row)
    if not comment_col:
        return ""

    return norm(ws.cell(row, comment_col).value)


def extract_json(file_path):
    wb = load_workbook(file_path, data_only=True)

    if TARGET_SHEET not in wb.sheetnames:
        raise ValueError(f"Không tìm thấy sheet '{TARGET_SHEET}'")

    ws = wb[TARGET_SHEET]

    data = {
        "name": get_name(wb),
        "quality_comment": get_comment_by_label(ws, "Yêu nước"),
        "general_abilities_comment": get_comment_by_label(ws, "Tự chủ và tự học"),
        "special_abilities_comment": get_comment_by_label(ws, "Ngôn ngữ"),
    }

    return data

In [4]:
from src.adapter.database.postgres_repository import PostgresStudentRepository
from src.adapter.database.postges_manager import postgres_manager
student_repo = PostgresStudentRepository(postgres_manager.session)

In [5]:
(student_repo.exists({"name": 'Võ Minh Khang'})).code

2026-04-12 21:54:36,007 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-12 21:54:36,009 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-12 21:54:36,245 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-12 21:54:36,247 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-12 21:54:36,487 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-12 21:54:36,490 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-12 21:54:36,725 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-12 21:54:36,731 INFO sqlalchemy.engine.Engine SELECT student.code AS student_code, student.name AS student_name, student.card_id AS student_card_id, student.edu_id AS student_edu_id, student.date_of_birth AS student_date_of_birth, student.gender AS student_gender, student.ethnicity AS student_ethnicity, student.status AS student_status, student.phone AS student_phone, student.nationality AS student_nationality, student.address AS student_address, student.place_of_birth

'20252026.01.Mo1.011'

In [6]:
from src.adapter.database.postges_manager import postgres_manager
from src.adapter.database.mongo_manager import mongo_manager
from src.adapter.graph.neo4j_manager import neo4j_manager
from src.application.core import SystemCore

session = postgres_manager.session
db = mongo_manager.get_metadata_db()
manager = neo4j_manager
repo = SystemCore(session, db, manager)

async def add_comment(payload: AssessmentComment):
    if not isinstance(payload.code, str):
       print("Invalid ID")
    # repo.clear_student_assessment_outstanding(payload.code)
    result = None
    if payload.quality_comment:
        result = await repo.add_student_comment(Comment(
            code=payload.code,
            comment=payload.quality_comment
        ), "Phẩm chất chủ yếu")
    if payload.general_abilities_comment:
        result = await repo.add_student_comment(Comment(
            code=payload.code,
            comment=payload.general_abilities_comment
        ), "Năng lực chung")
    if payload.special_abilities_comment:
        result = await repo.add_student_comment(Comment(
            code=payload.code,
            comment=payload.special_abilities_comment
        ), "Năng lực đặc thù")
    if not result:
        print(f"comment is not useful for {payload.code}, please check logs")
    return result


d:\Work\Iu\hoc-ba-so\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1540.59it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
cnt = 0
for file in excel_files:
    info = extract_json(file)
    comment = AssessmentComment(
        code = (student_repo.exists({"name": info.get("name")})).code,
        quality_comment=info.get("quality_comment"),
        general_abilities_comment=info.get("general_abilities_comment"),
        special_abilities_comment=info.get("special_abilities_comment")
    )
    await add_comment(comment)
    # cnt = cnt + 1
    # if cnt == 1:
    #     break

2026-04-12 21:56:03,955 INFO sqlalchemy.engine.Engine SELECT student.code AS student_code, student.name AS student_name, student.card_id AS student_card_id, student.edu_id AS student_edu_id, student.date_of_birth AS student_date_of_birth, student.gender AS student_gender, student.ethnicity AS student_ethnicity, student.status AS student_status, student.phone AS student_phone, student.nationality AS student_nationality, student.address AS student_address, student.place_of_birth AS student_place_of_birth 
FROM student 
WHERE student.name = %(name_1)s 
 LIMIT %(param_1)s
2026-04-12 21:56:03,956 INFO sqlalchemy.engine.Engine [cached since 87.22s ago] {'name_1': 'Võ Minh Khang', 'param_1': 1}
2026-04-12 21:56:04,081 INFO sqlalchemy.engine.Engine SELECT student.code AS student_code, student.name AS student_name, student.card_id AS student_card_id, student.edu_id AS student_edu_id, student.date_of_birth AS student_date_of_birth, student.gender AS student_gender, student.ethnicity AS student_e